# Eksperimen SML - Rafli Ardiansyah

Dataset: UCI Default of Credit Card Clients.

Notebook ini mengikuti alur eksperimen manual: data loading, EDA, preprocessing, dan export data siap latih.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib
import json
from pathlib import Path

## Data Loading

In [ ]:
DATA_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls'
df = pd.read_excel(DATA_URL, header=1)
df.head()

In [ ]:
df.info()
df.describe().T

## EDA

In [ ]:
df.columns = [str(c).strip().lower().replace(' ', '_').replace('-', '_') for c in df.columns]
df = df.rename(columns={'default_payment_next_month': 'default_payment_next_month'})
target = 'default_payment_next_month'
df[target].value_counts(normalize=True).plot(kind='bar', title='Target distribution')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='LIMIT_BAL'.lower(), hue=target, bins=40, element='step')
plt.title('Limit balance distribution by target')
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df.drop(columns=['id'], errors='ignore').corr(numeric_only=True), cmap='Blues', center=0)
plt.title('Correlation heatmap')
plt.show()

## Preprocessing Manual

In [ ]:
df_clean = df.drop(columns=['id'], errors='ignore').drop_duplicates().copy()
for col in df_clean.columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
df_clean = df_clean.dropna(subset=[target])
df_clean[target] = df_clean[target].astype(int)
categorical_cols = [c for c in ['sex', 'education', 'marriage'] if c in df_clean.columns]
numeric_cols = [c for c in df_clean.columns if c not in categorical_cols + [target]]
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode().iloc[0]).astype(int).astype(str)
for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

In [ ]:
X = df_clean.drop(columns=[target])
y = df_clean[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
preprocessor = ColumnTransformer([
    ('numeric', StandardScaler(), numeric_cols),
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
])
X_train_arr = preprocessor.fit_transform(X_train)
X_test_arr = preprocessor.transform(X_test)
feature_names = list(preprocessor.get_feature_names_out())

## Export Dataset Siap Latih

In [ ]:
output_dir = Path('../../SMSML_Rafli-Ardiansyah/Membangun_model/credit_default_preprocessing')
output_dir.mkdir(parents=True, exist_ok=True)
train = pd.DataFrame(X_train_arr, columns=feature_names)
train[target] = y_train.to_numpy()
test = pd.DataFrame(X_test_arr, columns=feature_names)
test[target] = y_test.to_numpy()
train.to_csv(output_dir / 'train.csv', index=False)
test.to_csv(output_dir / 'test.csv', index=False)
joblib.dump(preprocessor, output_dir / 'preprocessor.joblib')
metadata = {'rows_after_cleaning': len(df_clean), 'train_rows': len(train), 'test_rows': len(test), 'feature_count': len(feature_names)}
(output_dir / 'preprocessing_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
metadata